## 3. IT Notice Decomposition: From Conservative Estimate to Empirical Reality

The baseline model initially assumed a **12-hour conservative engineering notice
period** for flexible AI training workloads. This was a worst-case legacy assumption
designed to ensure the model did not over-promise flexibility.

Reconciliation against current hyperscaler documentation and NESO Balancing Mechanism
protocols reveals that modern distributed training infrastructure operates on a
dramatically faster timescale. The empirical reality is a **~1.0 hour total notice
period**.

| Phase | Conservative Baseline | Empirical Reality (2024-25) | Evidence Source |
| :--- | ---: | ---: | :--- |
| Checkpoint Write | 1.0 h | **0.08 h** (~5 min) | Async checkpointing, 200GB @ 4GB/s NVMe |
| Orchestration & Cooling | 1.5 h | **0.17 h** (~10 min) | AWS HyperPod / Azure ML automated drain |
| Grid Protocol | 2.5 h | **0.33 h** (~20 min) | NESO BM 20-min contractual delivery window |
| Restart Verification | 3.0 h | **0.17 h** (~10 min) | Shard reload + integrity check |
| Safety Margin | 4.0 h | **0.25 h** (~15 min) | Network latency / partial failures |
| **TOTAL** | **12.0 h** | **1.0 h** | |

**Policy implication:** The "Timescale Mismatch Trap" is isolated to P10 events
(<1 hour), which are too short for *any* large-scale physical response. Median
Scottish constraint events (1.5-2.0 hours) are fully compatible with modern
checkpointing. The bottleneck is software implementation, not physics.

In [1]:
# Cell 1: Environment verification and project root discovery
import sys
from pathlib import Path

def find_project_root() -> Path:
    """Walk up from cwd until we find pyproject.toml (the project sentinel)."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError(
        f"Could not find project root (no pyproject.toml found).\n"
        f"Searched from: {current}\n"
        f"Run `uv run jupyter lab` from the project root, not from notebooks/."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
CONFIGS_DIR = PROJECT_ROOT / "configs"

# Verify Python executable is from the project venv (Manifesto §4.1)
venv_python = Path(sys.executable)
assert "scotland-ai-split-zones" in str(venv_python), (
    f"Wrong Python executable: {venv_python}\n"
    f"Run notebooks via `uv run jupyter lab` from the project root."
)

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Python:        {venv_python}")
print(f"✅ Data dir:      {DATA_DIR}")
print(f"✅ Configs dir:   {CONFIGS_DIR}")

✅ Project root: /home/ndrew/scotland-ai-split-zones
✅ Python:        /home/ndrew/scotland-ai-split-zones/.venv/bin/python3
✅ Data dir:      /home/ndrew/scotland-ai-split-zones/data
✅ Configs dir:   /home/ndrew/scotland-ai-split-zones/configs


In [3]:
import polars as pl

# Inline scenario data (empirically grounded from Milestone 2.3 research)
# This avoids file I/O path mismatches and makes the notebook self-contained.
scenarios_data = [
    {"label": "Current (2024-25)", "total_notice_h": 1.0, "event_duration_h": 2.0, "compatibility_factor": 1.0, "confidence": "high"},
    {"label": "Near-term (2026-27)", "total_notice_h": 0.7, "event_duration_h": 2.0, "compatibility_factor": 1.0, "confidence": "medium"},
    {"label": "Medium-term (2028-30)", "total_notice_h": 0.4, "event_duration_h": 2.0, "compatibility_factor": 1.0, "confidence": "medium-low"},
    {"label": "Optimistic (2030+)", "total_notice_h": 0.25, "event_duration_h": 2.0, "compatibility_factor": 1.0, "confidence": "low"},
]

scenarios_df = pl.DataFrame(scenarios_data)

print("Duration Compatibility Factor — Median Scottish Event (2.0 hours)")
print("=" * 70)
print(scenarios_df.select(["label", "total_notice_h", "compatibility_factor", "confidence"]))
print()
print("Interpretation:")
print("  • Current (1.0h notice):  factor = 1.00  → event fully utilised")
print("  • Legacy (12.0h notice):  factor = 0.17  → 83% of value lost to mismatch")
print()
print("The empirical baseline captures the full value of the median event.")
print("The legacy assumption would have understated flexibility by ~6×.")

Duration Compatibility Factor — Median Scottish Event (2.0 hours)
shape: (4, 4)
┌───────────────────────┬────────────────┬──────────────────────┬────────────┐
│ label                 ┆ total_notice_h ┆ compatibility_factor ┆ confidence │
│ ---                   ┆ ---            ┆ ---                  ┆ ---        │
│ str                   ┆ f64            ┆ f64                  ┆ str        │
╞═══════════════════════╪════════════════╪══════════════════════╪════════════╡
│ Current (2024-25)     ┆ 1.0            ┆ 1.0                  ┆ high       │
│ Near-term (2026-27)   ┆ 0.7            ┆ 1.0                  ┆ medium     │
│ Medium-term (2028-30) ┆ 0.4            ┆ 1.0                  ┆ medium-low │
│ Optimistic (2030+)    ┆ 0.25           ┆ 1.0                  ┆ low        │
└───────────────────────┴────────────────┴──────────────────────┴────────────┘

Interpretation:
  • Current (1.0h notice):  factor = 1.00  → event fully utilised
  • Legacy (12.0h notice):  factor = 0.17  → 83